In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass

import math

torch.manual_seed(2006)

In [5]:
@dataclass
class GPTConfig:
    block_size: int = 512 # 就是seq_len（max_seq_len）
    batch_size: int = 12
    n_layer: int = 6
    n_head: int = 12
    n_embed: int = 768 # 默认了hidden_dim和这个一个维度了
    head_size: int = n_embed // n_head
    dropout: float = 0.1
    # tiktoken 使用的是GPT-2的词表，大约有50257个token
    vocab_size: int = 50257


##### 1.register_buffer

*好处*:
- 不会随着模型训练更新(与 nn.Parameter() 区别，有参数presistent控制)
- 能够随着 model.to("cuda") 自动切换到GPU
- 能够直接跟着保存权重 state_dict


##### 2.masked_fill

- tensor.masked_fill(mask, value)

其中，mask:掩码矩阵，值为1或者True的直接被覆盖;value:覆盖在选定位置的值，如:-inf

一般与 torch.tril(torch.ones(size=(some_matrix_shape))) 搭配使用

In [6]:
class SingleHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()

        # 实际上后面的n_embed就是hidden_dim，只是这里正好相等
        self.q = nn.Linear(config.n_embed, config.head_size)
        self.k = nn.Linear(config.n_embed, config.head_size)
        self.v = nn.Linear(config.n_embed, config.head_size)

        # tril: 以“l”结尾，就是“lower”，i.e. 1的下三角矩阵
        # triu: 以“u”结尾，就是“upper”，i.e. 1的上三角矩阵
        self.register_buffer(
            "attention_mask",
            torch.tril(
                torch.ones(config.block_size, config.block_size)
            )
        )

        self.dropout = nn.Dropout(config.dropout)
        # self.softmax = nn.Softmax()

    attention_mask: torch.Tensor

    def forward(self, x):
        batch_size, seq_len, hidden_size = x.size()

        # [batch_size, seq_len, head_size]
        Q = self.q(x)
        K = self.k(x)
        V = self.v(x)

        # [batch_size, seq_len, seq_len]
        attn = Q @ K.transpose(-2, -1)

        # 经过 == 0 变成了1的上三角矩阵，填入 float("-inf")
        attn = attn.masked_fill(
            self.attention_mask[:seq_len, :seq_len] == 0,
            float('-inf')
        ) / math.sqrt(hidden_size)

        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)

        # [batch_size, seq_len, head_size]
        x = attn @ V

        return x

In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.proj = nn.Linear(config.n_embed, config.n_embed)
        self.nets = nn.ModuleList([
            SingleHeadAttention(config) for _ in range(config.n_head)
        ])
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        # [batch_size, seq_len, n_embed] 
        x = torch.cat(
            [h(x) for h in self.nets],
            dim = -1
        )

        x = self.proj(x)
        x = self.dropout(x)

        return x

In [8]:
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.n_embed, 4 * config.n_embed),
            nn.GELU(),
            nn.Linear(4 * config.n_embed, config.n_embed),
            nn.Dropout(config.dropout)
        )

    def forward(self, x):
        return self.net(x)

In [9]:
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.att = MultiHeadAttention(config)
        self.ffn = FeedForward(config)
        self.ln1 = nn.LayerNorm(config.n_embed)
        self.ln2 = nn.LayerNorm(config.n_embed)

    def forward(self, x):
        x = x + self.att(self.ln1(x))
        x = x + self.ffn(self.ln2(x))

        return x

In [10]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.token_embedding_table = nn.Embedding(config.vocab_size, config.n_embed)
        self.position_embedding_table = nn.Embedding(config.block_size, config.n_embed)
        self.blocks = nn.Sequential(
            *[Block(config) for _ in range(config.n_layer)]
        )
        self.ln_final = nn.LayerNorm(config.n_embed)
        self.lm_head = nn.Linear(config.n_embed, config.vocab_size, bias=False)

        self.block_size : int = config.block_size

        self.apply(self._init_weights)

    def _init_weights(self, module):
        # isinstance(a, b)检查a是否是b的实例或者是子类
        if isinstance(module, nn.Linear):
            # 使用正态分布初始化
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        # idx是输入的token_ids
        batch, seq_len = idx.size()
        token_emb = self.token_embedding_table(idx)

        #seq长度是这次输入的最大长度
        pos_emb = self.position_embedding_table(
            # 确保位置编码与idx在同一个设备上
            torch.arange(seq_len, device=idx.device)
        )

        # [batch, seq_len, n_embed]
        x = token_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_final(x)

        # [batch, seq_len, vocab_size]
        logits = self.lm_head(x)

        if targets == None:
            loss = None
        else:
            batch, seq_len, vocab_size = logits.size()
            logits = logits.view(batch * seq_len, vocab_size)
            targets = targets.view(batch * seq_len)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx : [B, T]
        # max_new_tokens : 准备预测多少个词
        for _ in range(max_new_tokens):
            # 如果序列太长，只取最后block_size个token
            idx_cond = idx if idx.size(1) <= self.block_size else idx[:, -self.block_size:]

            # 获取预测
            # logits, _ = self.forward(idx_cond)
            logits, _ = self(idx_cond)

            # 只关注最后一个时间步的预测 [B, vocab_size]
            logits = logits[:, -1, :]

            # 应用softmax获取概率
            probs = F.softmax(logits, dim=-1)

            # 采样下一个token
            # Greedy search 太死板并且容易陷入死循环
            idx_next = torch.multinomial(probs, num_samples=1)

            # 附加到序列上
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

In [11]:
class  MyDataset(Dataset):
    def __init__(self, path, block_size=512):

        import tiktoken
        self.enc = tiktoken.get_encoding("gpt2")
        self.block_size = block_size

        self.eos_token = self.enc.encode(
            "<|endoftext|>",
            allowed_special={"<|endoftext|>"}
        )[0]

        import json

        self.encoded_data = []
        self.max_lines = 1000
        raw_data = []

        with open(path, 'r') as f:
            for i, line in enumerate(f):
                if i >= self.max_lines:
                    break
                try:
                    text = json.loads(line.strip())['text']
                except json.JSONDecodeError:
                    continue
                except Exception as e:
                    continue

        full_encoded = []
        for text in raw_data:
            encoded_text = self.enc.encode(text)
            full_encoded.extend(encoded_text + [self.eos_token])

        # 将长文本分割成训练样本
        for i in range(0, len(full_encoded), self.block_size):
            # 多取一个token作为目标
            chunk = full_encoded[i:i+self.block_size+1]

            # 如果最后一块长度不够，用eos_token填充
            if len(chunk) < self.block_size + 1:
                chunk = chunk + [self.eos_token] * (self.block_size + 1 - len(chunk))

            self.encoded_data.append(chunk)

    def __len__(self):
        return len(self.encoded_data)

    def __getitem__(self, idx):
        chunk = self.encoded_data[idx]
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)

        return x, y

    def encode(self, text):
        # 将文本编码为 token IDs
        return self.enc.encode(text)

    def decode(self, ids):
        # 将 token IDs 解码为文本
        return self.enc.decode(ids)

In [12]:
# 数据的格式
"""
{"text":"担任地点省市的区域运营中心的办理作业。承受总部相关KPI查核。\n1、了解新闻职业或媒体相关运营运营岗位，其间，应聘区域运营中心主任有3年以上当地干流媒体作业经验者优先，应聘事务主管有2年以上当地干流媒体作业经验者优先。\n2、交流才能强，抗压才能强，长于处理复杂情况，了解GR作业优先，能独立完结策划计划优先。具有独立开发客户才能。\n北京、天津、河北、山西、黑龙江、吉林、辽宁、上海、江苏、浙江、安徽、江西、福建、山东、河南、湖北、湖南、广东、海南、重庆、四川、贵州、云南、陕西等。"}
"""

'\n{"text":"担任地点省市的区域运营中心的办理作业。承受总部相关KPI查核。\n1、了解新闻职业或媒体相关运营运营岗位，其间，应聘区域运营中心主任有3年以上当地干流媒体作业经验者优先，应聘事务主管有2年以上当地干流媒体作业经验者优先。\n2、交流才能强，抗压才能强，长于处理复杂情况，了解GR作业优先，能独立完结策划计划优先。具有独立开发客户才能。\n北京、天津、河北、山西、黑龙江、吉林、辽宁、上海、江苏、浙江、安徽、江西、福建、山东、河南、湖北、湖南、广东、海南、重庆、四川、贵州、云南、陕西等。"}\n'

In [13]:
# train data
train_dataset = MyDataset()

# split traindataset to train and val
train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [0.9, 0.1])

train_loader = DataLoader(train_dataset, batch_size=12, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=12, shuffle=True)

TypeError: MyDataset.__init__() missing 1 required positional argument: 'path'

In [ ]:
model = GPT(GPTConfig)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# 模型参数计算

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params / 1e-6 } M")

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

# 设置consine学习率
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1000)

In [ ]:
def train(model, optimizer, scheduler, train_loader, device, epoch):
    model.train()
    total_loss = 0
    for batch_idx, (x, y) in enumerate(train_loader):
        # 将数据转移到设备上
        x, y = x.to(device), y.to(device)

        # 前向传播
        logits, loss = model(x, targets=y)

        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # 调整学习率
        scheduler.step()

        total_loss += loss.item()

        if batch_idx % 100 == 0:
            print(f"Epoch:{epoch}, Batch:{batch_idx}, Loss:{loss.item():.4f}")

        return total_loss

In [ ]:
def eval(model, val_loader, device):
    # 验证
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits, loss = model(x, targets=y)
            val_loss += loss.item()
    return val_loss

In [ ]:
for epoch in range(2):
    train_loss = train(model, optimizer, scheduler, train_loader, device, epoch)
    val_loss = eval(model, val_loader, device)
    print(f"Epoch:{epoch}, Train Loss:{train_loss/len(train_loader):.4f}, Val Loss:{val_loss/len(val_loader):.4f}")

    # 保存模型
    avg_val_loss = val_loss / len(val_loader)
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'val_loss': avg_val_loss,
    }

    # 保存每个epoch的模型
    torch.save(checkpoint, f'checkpoints/model_epoch_{epoch}.pt')